In [2]:
#!rm -rf /workspace/18-4-2026/experiments/2026/15-4-2026/activations

In [3]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
REPO_DIR = "/workspace/18-4-2026"
EXP_DIR = os.path.join(REPO_DIR, "experiments", "2026", "15-4-2026")

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(EXP_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory so Path.cwd().parent resolves to experiment root
os.chdir(os.path.join(EXP_DIR, "notebooks"))

HEAD is now at 2ab22ca Add classification results for all 5 transform strategies


# NB1: Generator Activation Extraction (H200 GPU)

Load G3 (QwQ-32B) and G1 (DeepSeek-R1-Distill-Qwen-32B) sequentially on H200,
extract residual-stream activations for Phase 2 analysis.

**Outputs:**
- `activations/{G1,G3}_last_token/` -- last-token pooled activations at every 4th layer
- `activations/{G1,G3}_question_token/` -- pre-`<think>` position activations (for Exp B)
- `activations/{G1,G3}_full_seq/` -- full-sequence activations at 8 selected layers (for CKA)

**GPU time:** ~3 hours (1.5h per model, sequential)

In [4]:
import sys
import json
import gc
import torch
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_phase1_results, get_labeled_cots, join_cots_with_labels,
    get_within_question_pairs, load_model, extract_activations,
    extract_activations_at_position, save_activations, load_activations,
    get_default_layer_indices, print_phase1_summary,
    LOCAL_MODELS, ACTIVATIONS_DIR,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA H200
VRAM: 150.0 GB


In [5]:
# Load Phase 1 results and CoT texts
print_phase1_summary()

# Get all classified (non-filtered) CoTs with text for G1 and G3
# We extract activations for ALL classified samples (leaked + legible + illegible)
# so probes can be trained on any split
all_cots_g1 = join_cots_with_labels(generator_ids=["G1"])
all_cots_g3 = join_cots_with_labels(generator_ids=["G3"])

print(f"G1 CoTs: {len(all_cots_g1)}")
print(f"G3 CoTs: {len(all_cots_g3)}")

# Sort by sample_id for reproducible ordering
all_cots_g1 = sorted(all_cots_g1, key=lambda x: (x['sample_id'], x['epoch']))
all_cots_g3 = sorted(all_cots_g3, key=lambda x: (x['sample_id'], x['epoch']))

Total records: 1285
Classified: 666, Filtered: 619
R4 transform: _t64
Label counts:
  ANSWER_LEAKED: 278
  FILTERED: 619
  ILLEGIBLE: 288
  REASONING_LEGIBLE: 100

Per-generator:
  G1: 219 classified (leaked=25%, legible=15%, illegible=60%)
  G2: 218 classified (leaked=69%, legible=17%, illegible=14%)
  G3: 229 classified (leaked=32%, legible=13%, illegible=55%)
  G1 within-Q pairs: 0
  G3 within-Q pairs: 0
G1 CoTs: 219
G3 CoTs: 229


In [6]:
# Verify CoT lengths and spot-check a few samples
for name, cots in [("G1", all_cots_g1), ("G3", all_cots_g3)]:
    lengths = [len(c['cot_text']) for c in cots]
    labels = [c['label'] for c in cots]
    from collections import Counter
    print(f"\n{name}:")
    print(f"  Label distribution: {dict(Counter(labels))}")
    print(f"  CoT length: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")
    # Spot check
    sample = cots[0]
    print(f"  Sample: {sample['sample_id']} label={sample['label']}")
    print(f"  CoT preview: {sample['cot_text'][:200]}...")


G1:
  Label distribution: {'ILLEGIBLE': 131, 'REASONING_LEGIBLE': 33, 'ANSWER_LEAKED': 55}
  CoT length: min=16, median=1092, max=2506
  Sample: gpqa_103 label=ILLEGIBLE
  CoT preview: Astronomers are searching for exoplanets around two stars with exactly the same masses. Using the RV method, they detected one planet around each star, both with masses similar to that of Neptune. The...

G3:
  Label distribution: {'ILLEGIBLE': 126, 'ANSWER_LEAKED': 73, 'REASONING_LEGIBLE': 30}
  CoT length: min=16, median=1239, max=2951
  Sample: gpqa_103 label=ILLEGIBLE
  CoT preview: To determine the ratio of the orbital periods of planet #2 to planet #1, we use the relationship between radial velocity semi-amplitude (\(K\)) and orbital period (\(P\)). The radial velocity \(K\) is...


## G3: QwQ-32B Activation Extraction

In [7]:
# Check if all G3 outputs already exist (skip model load if so)
g3_outputs = ["G3_last_token", "G3_question_token", "G3_full_seq"]
g3_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in g3_outputs)

if g3_all_cached:
    print("CACHED: All G3 outputs exist, skipping G3 model load")
    model_g3 = tok_g3 = None
    n_layers_g3 = None
else:
    # Load G3 (QwQ-32B) -- ~64GB VRAM at bf16
    model_g3, tok_g3 = load_model("G3")
    n_layers_g3 = model_g3.config.num_hidden_layers
    print(f"G3 layers: {n_layers_g3}, hidden_dim: {model_g3.config.hidden_size}")

Loading G3 (Qwen/QwQ-32B)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

  Loaded: 32.8B params, 4bit=False
G3 layers: 64, hidden_dim: 5120


In [8]:
output_dir = ACTIVATIONS_DIR / "G3_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_last_token = load_activations(output_dir)
else:
    # Extract last-token activations at every 4th layer + last
    layer_indices = get_default_layer_indices(n_layers_g3, stride=4)
    print(f"Extracting at layers: {layer_indices}")

    g3_texts = [c['cot_text'] for c in all_cots_g3]
    g3_last_token = extract_activations(
        model_g3, tok_g3, g3_texts,
        layer_indices=layer_indices,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in g3_last_token.items():
        assert acts.shape[0] == len(g3_texts), f"Layer {layer_idx}: expected {len(g3_texts)}, got {acts.shape[0]}"
        assert not np.any(np.isnan(acts)), f"Layer {layer_idx}: NaN detected"
        assert not np.any(np.isinf(acts)), f"Layer {layer_idx}: Inf detected"
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    # Save with metadata
    metadata_g3 = {
        "model": "G3",
        "hf_id": LOCAL_MODELS["G3"]["hf_id"],
        "pooling": "last_token",
        "n_samples": len(g3_texts),
        "layer_indices": layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g3],
    }
    save_activations(g3_last_token, output_dir, metadata=metadata_g3)

Extracting at layers: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 63]


Extracting activations: 100%|██████████| 229/229 [00:18<00:00, 12.17it/s]


  Layer 0: shape=(229, 5120), norm=1.60
  Layer 4: shape=(229, 5120), norm=63.57
  Layer 8: shape=(229, 5120), norm=89.60
  Layer 12: shape=(229, 5120), norm=128.71
  Layer 16: shape=(229, 5120), norm=139.58
  Layer 20: shape=(229, 5120), norm=140.47
  Layer 24: shape=(229, 5120), norm=148.82
  Layer 28: shape=(229, 5120), norm=159.71
  Layer 32: shape=(229, 5120), norm=173.22
  Layer 36: shape=(229, 5120), norm=172.47
  Layer 40: shape=(229, 5120), norm=176.50
  Layer 44: shape=(229, 5120), norm=204.50
  Layer 48: shape=(229, 5120), norm=278.74
  Layer 52: shape=(229, 5120), norm=392.09
  Layer 56: shape=(229, 5120), norm=545.48
  Layer 60: shape=(229, 5120), norm=760.44
  Layer 63: shape=(229, 5120), norm=1585.90
  Saved 17 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G3_last_token


In [9]:
output_dir = ACTIVATIONS_DIR / "G3_question_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_question_token = load_activations(output_dir)
else:
    # Extract pre-<think> activations for Experiment B
    # Build input texts: question + <think> tag (the model sees this before generating CoT)
    layer_indices = get_default_layer_indices(n_layers_g3, stride=4)
    g3_question_texts = []
    for c in all_cots_g3:
        # Reconstruct the input as the model would have seen it
        g3_question_texts.append(c['input'] + "\n<think>")

    g3_question_token = extract_activations_at_position(
        model_g3, tok_g3, g3_question_texts,
        position="pre_think",
        layer_indices=layer_indices,
        max_length=4096,  # Questions are short
    )

    # Verify
    for layer_idx, acts in g3_question_token.items():
        assert acts.shape[0] == len(g3_question_texts)
        assert not np.any(np.isnan(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_g3 = {
        "model": "G3",
        "hf_id": LOCAL_MODELS["G3"]["hf_id"],
        "pooling": "pre_think_position",
        "n_samples": len(g3_question_texts),
        "layer_indices": layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g3],
    }
    save_activations(g3_question_token, output_dir, metadata=metadata_g3)

Extracting at pre_think: 100%|██████████| 229/229 [00:10<00:00, 20.82it/s]


  Layer 0: shape=(229, 5120)
  Layer 4: shape=(229, 5120)
  Layer 8: shape=(229, 5120)
  Layer 12: shape=(229, 5120)
  Layer 16: shape=(229, 5120)
  Layer 20: shape=(229, 5120)
  Layer 24: shape=(229, 5120)
  Layer 28: shape=(229, 5120)
  Layer 32: shape=(229, 5120)
  Layer 36: shape=(229, 5120)
  Layer 40: shape=(229, 5120)
  Layer 44: shape=(229, 5120)
  Layer 48: shape=(229, 5120)
  Layer 52: shape=(229, 5120)
  Layer 56: shape=(229, 5120)
  Layer 60: shape=(229, 5120)
  Layer 63: shape=(229, 5120)
  Saved 17 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G3_question_token


In [10]:
output_dir = ACTIVATIONS_DIR / "G3_full_seq"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g3_full_seq = load_activations(output_dir)
else:
    # Extract full-sequence activations at 8 selected layers for CKA analysis
    # Only for legible + illegible samples (not answer-leaked) to save space
    cka_layer_indices = [0, 8, 16, 24, 32, 40, 48, 63]

    g3_non_leaked = [c for c in all_cots_g3 if c['label'] != 'ANSWER_LEAKED']
    g3_non_leaked_texts = [c['cot_text'] for c in g3_non_leaked]
    print(f"Extracting full-seq for {len(g3_non_leaked_texts)} non-leaked G3 CoTs")
    print(f"Layers: {cka_layer_indices}")

    g3_full_seq = extract_activations(
        model_g3, tok_g3, g3_non_leaked_texts,
        layer_indices=cka_layer_indices,
        pooling="full",
        batch_size=1,
        max_length=16384,
    )

    # Verify
    for layer_idx, acts_list in g3_full_seq.items():
        print(f"  Layer {layer_idx}: {len(acts_list)} sequences, "
              f"seq_lens=[{acts_list[0].shape[0]}..{acts_list[-1].shape[0]}]")

    metadata_g3_full = {
        "model": "G3",
        "pooling": "full_sequence",
        "n_samples": len(g3_non_leaked_texts),
        "layer_indices": cka_layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in g3_non_leaked],
    }
    save_activations(g3_full_seq, output_dir, metadata=metadata_g3_full)

Extracting full-seq for 156 non-leaked G3 CoTs
Layers: [0, 8, 16, 24, 32, 40, 48, 63]


Extracting activations: 100%|██████████| 156/156 [00:14<00:00, 10.62it/s]


  Layer 0: 156 sequences, seq_lens=[292..520]
  Layer 8: 156 sequences, seq_lens=[292..520]
  Layer 16: 156 sequences, seq_lens=[292..520]
  Layer 24: 156 sequences, seq_lens=[292..520]
  Layer 32: 156 sequences, seq_lens=[292..520]
  Layer 40: 156 sequences, seq_lens=[292..520]
  Layer 48: 156 sequences, seq_lens=[292..520]
  Layer 63: 156 sequences, seq_lens=[292..520]
  Saved 8 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G3_full_seq


In [11]:
# Unload G3 before loading G1
if model_g3 is not None:
    del model_g3, tok_g3
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after unload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

GPU memory after unload: 0.0 GB


## G1: DeepSeek-R1-Distill-Qwen-32B Activation Extraction

In [12]:
# Check if all G1 outputs already exist (skip model load if so)
g1_outputs = ["G1_last_token", "G1_question_token", "G1_full_seq"]
g1_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in g1_outputs)

if g1_all_cached:
    print("CACHED: All G1 outputs exist, skipping G1 model load")
    model_g1 = tok_g1 = None
    n_layers_g1 = None
else:
    # Load G1 (DeepSeek-R1-Distill-Qwen-32B) -- ~64GB VRAM at bf16
    model_g1, tok_g1 = load_model("G1")
    n_layers_g1 = model_g1.config.num_hidden_layers
    print(f"G1 layers: {n_layers_g1}, hidden_dim: {model_g1.config.hidden_size}")

Loading G1 (deepseek-ai/DeepSeek-R1-Distill-Qwen-32B)...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

  Loaded: 32.8B params, 4bit=False
G1 layers: 64, hidden_dim: 5120


In [13]:
output_dir = ACTIVATIONS_DIR / "G1_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_last_token = load_activations(output_dir)
else:
    # Extract last-token activations
    layer_indices_g1 = get_default_layer_indices(n_layers_g1, stride=4)
    print(f"Extracting at layers: {layer_indices_g1}")

    g1_texts = [c['cot_text'] for c in all_cots_g1]
    g1_last_token = extract_activations(
        model_g1, tok_g1, g1_texts,
        layer_indices=layer_indices_g1,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in g1_last_token.items():
        assert acts.shape[0] == len(g1_texts)
        assert not np.any(np.isnan(acts))
        assert not np.any(np.isinf(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    metadata_g1 = {
        "model": "G1",
        "hf_id": LOCAL_MODELS["G1"]["hf_id"],
        "pooling": "last_token",
        "n_samples": len(g1_texts),
        "layer_indices": list(layer_indices_g1),
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g1],
    }
    save_activations(g1_last_token, output_dir, metadata=metadata_g1)

Extracting at layers: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 63]


Extracting activations: 100%|██████████| 219/219 [00:16<00:00, 13.35it/s]


  Layer 0: shape=(219, 5120), norm=1.61
  Layer 4: shape=(219, 5120), norm=88.21
  Layer 8: shape=(219, 5120), norm=126.19
  Layer 12: shape=(219, 5120), norm=184.38
  Layer 16: shape=(219, 5120), norm=194.29
  Layer 20: shape=(219, 5120), norm=193.69
  Layer 24: shape=(219, 5120), norm=199.64
  Layer 28: shape=(219, 5120), norm=208.13
  Layer 32: shape=(219, 5120), norm=230.71
  Layer 36: shape=(219, 5120), norm=236.85
  Layer 40: shape=(219, 5120), norm=251.66
  Layer 44: shape=(219, 5120), norm=292.89
  Layer 48: shape=(219, 5120), norm=394.32
  Layer 52: shape=(219, 5120), norm=554.91
  Layer 56: shape=(219, 5120), norm=763.39
  Layer 60: shape=(219, 5120), norm=1051.32
  Layer 63: shape=(219, 5120), norm=2111.66
  Saved 17 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G1_last_token


In [14]:
output_dir = ACTIVATIONS_DIR / "G1_question_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_question_token = load_activations(output_dir)
else:
    # Extract pre-<think> activations for Experiment B
    layer_indices_g1 = get_default_layer_indices(n_layers_g1, stride=4)
    g1_question_texts = [c['input'] + "\n<think>" for c in all_cots_g1]

    g1_question_token = extract_activations_at_position(
        model_g1, tok_g1, g1_question_texts,
        position="pre_think",
        layer_indices=layer_indices_g1,
        max_length=4096,
    )

    for layer_idx, acts in g1_question_token.items():
        assert acts.shape[0] == len(g1_question_texts)
        assert not np.any(np.isnan(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_g1_q = {
        "model": "G1",
        "hf_id": LOCAL_MODELS["G1"]["hf_id"],
        "pooling": "pre_think_position",
        "n_samples": len(g1_question_texts),
        "layer_indices": list(layer_indices_g1),
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in all_cots_g1],
    }
    save_activations(g1_question_token, output_dir, metadata=metadata_g1_q)

Extracting at pre_think: 100%|██████████| 219/219 [00:11<00:00, 19.09it/s]


  Layer 0: shape=(219, 5120)
  Layer 4: shape=(219, 5120)
  Layer 8: shape=(219, 5120)
  Layer 12: shape=(219, 5120)
  Layer 16: shape=(219, 5120)
  Layer 20: shape=(219, 5120)
  Layer 24: shape=(219, 5120)
  Layer 28: shape=(219, 5120)
  Layer 32: shape=(219, 5120)
  Layer 36: shape=(219, 5120)
  Layer 40: shape=(219, 5120)
  Layer 44: shape=(219, 5120)
  Layer 48: shape=(219, 5120)
  Layer 52: shape=(219, 5120)
  Layer 56: shape=(219, 5120)
  Layer 60: shape=(219, 5120)
  Layer 63: shape=(219, 5120)
  Saved 17 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G1_question_token


In [15]:
output_dir = ACTIVATIONS_DIR / "G1_full_seq"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    g1_full_seq = load_activations(output_dir)
else:
    # Full-sequence activations for CKA (non-leaked only)
    cka_layer_indices = [0, 8, 16, 24, 32, 40, 48, 63]

    g1_non_leaked = [c for c in all_cots_g1 if c['label'] != 'ANSWER_LEAKED']
    g1_non_leaked_texts = [c['cot_text'] for c in g1_non_leaked]
    print(f"Extracting full-seq for {len(g1_non_leaked_texts)} non-leaked G1 CoTs")

    g1_full_seq = extract_activations(
        model_g1, tok_g1, g1_non_leaked_texts,
        layer_indices=cka_layer_indices,
        pooling="full",
        batch_size=1,
        max_length=16384,
    )

    for layer_idx, acts_list in g1_full_seq.items():
        print(f"  Layer {layer_idx}: {len(acts_list)} sequences")

    metadata_g1_full = {
        "model": "G1",
        "pooling": "full_sequence",
        "n_samples": len(g1_non_leaked_texts),
        "layer_indices": cka_layer_indices,
        "sample_ids": [(c['sample_id'], c['generator_id'], c['epoch'], c['label']) for c in g1_non_leaked],
    }
    save_activations(g1_full_seq, output_dir, metadata=metadata_g1_full)

Extracting full-seq for 164 non-leaked G1 CoTs


Extracting activations: 100%|██████████| 164/164 [00:17<00:00,  9.49it/s]


  Layer 0: 164 sequences
  Layer 8: 164 sequences
  Layer 16: 164 sequences
  Layer 24: 164 sequences
  Layer 32: 164 sequences
  Layer 40: 164 sequences
  Layer 48: 164 sequences
  Layer 63: 164 sequences
  Saved 8 layers to /workspace/18-4-2026/experiments/2026/15-4-2026/activations/G1_full_seq


In [16]:
# Final cleanup
if model_g1 is not None:
    del model_g1, tok_g1
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("NB1 complete.")

GPU memory after cleanup: 0.0 GB
NB1 complete.


In [17]:
# Summary of saved files
import os
for subdir in ['G1_last_token', 'G1_question_token', 'G1_full_seq',
               'G3_last_token', 'G3_question_token', 'G3_full_seq']:
    path = ACTIVATIONS_DIR / subdir
    if path.exists():
        total_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
        print(f"  {subdir}: {total_size / 1e9:.2f} GB")
    else:
        print(f"  {subdir}: NOT FOUND")

  G1_last_token: 0.08 GB
  G1_question_token: 0.08 GB
  G1_full_seq: 12.28 GB
  G3_last_token: 0.08 GB
  G3_question_token: 0.08 GB
  G3_full_seq: 13.39 GB
